# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

## 0. 환경 설정 및 준비

In [115]:
from dotenv import load_dotenv
load_dotenv()

True

In [116]:
import os
from pprint import pprint

## 1. 문서 준비/로드
### PDF

In [117]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('.././data/membersystem_access_rules.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

PDF 문서 개수: 15


In [118]:
# 첫 번째 문서의 내용 출력
print(pdf_docs[0].page_content)

- 1 -
회원시스템접속등에관한지침제정 2011. 12. 28. 기준 제265호개정 2012. 4. 27. 기준 제284호개정 2013. 5. 3. 기준 제328호개정 2013. 9. 16. 기준 제350호개정 2014. 2. 28. 기준 제377호개정 2014. 3. 20. 기준 제381호개정 2015. 4. 13. 기준 제442호개정 2015. 8. 7. 기준 제461호개정 2021. 6. 14. 지침 제739호개정 2022. 5. 18. 지침 제776호개정 2022. 12. 28. 지침 제813호개정 2025. 5. 30. 지침 제932호개정 2025. 11. 12. 지침 제(현행)968호개정 2025. 12. 15. 지침 제975호제1장 총칙제1조(목적) 이 지침은 한국거래소 「유가증권시장 업무규정」 제8조의2, 「코스닥시장 업무규정」 제7조의3, 「코넥스시장 업무규정」 제9조, 「파생상품시장 업무규정」 제8조의2, 「KRX금시장 운영규정」 제11조 및 「배출권 거래시장 운영규정」제10조의2에 따라 회원과 한국거래소 간 및 회원과 위탁자 간의 시스템 연결 등에 관하여 필요한 사항을 규정함을 목적으로 한다. <개정 2013. 9. 16., 2014. 3. 20., 2021. 6. 14.,2025. 11. 12.>제2조(정의) ① 이 지침에서 “회원”이란 한국거래소(이하 “거래소”라 한다) 「회원관리규정」 제3조제1항 각 호, 「KRX금시장 운영규정」제12조제1항 각 호 또는 「배출권 거래시장 운영규정」제11조의2제3호에 따른 회원을 말한다. <개정 2014. 3. 20., 2021. 6. 14.,2025. 11. 12.>② 이 지침에서 “거래소시스템”이란 거래소가 개설한 증권시장, 파생상품시장, KRX금시장 및 배출권 거래시장에서의 거래 등을 위하여 거래소가 운영하는 시스템을 말한다. <개정 2013. 9. 16.,


In [119]:
# 첫 번째 문서의 메타데이터 출력
pprint(pdf_docs[0].metadata)

{'author': '',
 'creationdate': '',
 'creator': 'Call PDF',
 'page': 0,
 'page_label': '1',
 'producer': 'Call PDF v 2.4',
 'source': '.././data/membersystem_access_rules.pdf',
 'subject': '',
 'title': '',
 'total_pages': 15}


### text

In [120]:
# TXT 파일 로드
with open(".././data/derivatives_rules_details.txt", "r") as f:
    text_content = f.read()

from langchain_core.documents import Document

# 단일 Document 객체 생성
text_doc = Document(
    page_content=text_content, 
    metadata={"source": "파생상품시장 업무규정 시행세칙_전문"}
)

In [121]:
# PDF와 TXT 문서 합치기
all_docs = pdf_docs + [text_doc]  # ⭐ 리스트로 감싸기!

print(f"전체 문서 개수: {len(all_docs)}")
print(f"  - PDF 문서: {len(pdf_docs)}개")
print(f"  - TXT 문서: 1개")
print(f"\n마지막 문서(TXT) 메타데이터:")
print(all_docs[-1].metadata)

전체 문서 개수: 16
  - PDF 문서: 15개
  - TXT 문서: 1개

마지막 문서(TXT) 메타데이터:
{'source': '파생상품시장 업무규정 시행세칙_전문'}


## 2. 텍스트 분할/문서 청킹 (Text Splitting)

**왜 필요한가?**
- PDF 15페이지 + 긴 TXT 파일 → 한 번에 임베딩하면 너무 큼
- LLM의 컨텍스트 윈도우 제한 (gpt-4o-mini: 128k 토큰)
- 작은 조각으로 나눠야 검색 정확도 ↑

**어떻게?**
- RecursiveCharacterTextSplitter 사용
- 법률 문서 특성: 조문 단위로 자르기 권장

**참고:**
- PRJ01_W2_004_Text_Splitter.ipynb
- chunk_size: 500-1000 (법률 문서는 500 권장)
- chunk_overlap: 100 (문맥 유지)

In [122]:
print(f'다섯 번째 문서: {all_docs[4]}')

다섯 번째 문서: page_content='- 5 -
<신설 2025. 12. 15.>⑤ 회원에게 배정하는 통신회선의 용량은 다음 각 호의 범위 내로 한다. <신설 2012. 4. 27., 2015. 4. 13., 2022. 5. 18.,개정 2025. 12. 15.>1. 증권시장의 경우 45M2. 파생상품시장의 경우 45M⑥ 2개 이상의 회원전산센터를 설치한 회원은 각 회원전산센터에 연결된 각 통신회선의 용량이 동일하도록 관리하여야 한다. <신설 2012. 4. 27.,개정 2025. 5. 30.>⑦ 거래소는 회원에게 배정할 수 있는 증권시장과 파생상품시장별 통신회선의 최대 용량을 정하거나 변경하는 경우 이를 즉시 회원에게 통지한다.제6조(통신회선의 신청) ① 회원은 제5조제2항 각 호에서 정한 통신회선의 범위 내에서 거래소에 통신회선의 신규배정, 추가배정 또는 변경배정을 신청할 수 있다. <개정 2012. 4. 27.,2025. 5. 30.>② 제1항에 따라 파생상품시장의 통신회선을 신청하는 회원은 회원전산센터별로 파생상품거래공유회선 또는 파생상품거래공유회선이 아닌 통신회선 중 하나로만 구성되도록 신청하여야 한다. <신설 2025. 5. 30.>③ 제1항의 신청은 별지 제1호 서식에 따라 통신회선 변경일 전 10거래일까지 거래소에 신청한다.④ 회원의 합병·분할·이전 등으로 인하여 회원시스템이 변경되는 경우의 통신회선의 배정 및 신청은 거래소가 그때마다 정하는 바에 따른다. <개정 2015. 8. 7.>제7조(네트워크의 연결) ① 회원은 거래소시스템과 연결하는 회원시스템의 네트워크를 변경하려는 경우 별지 제2호 서식에 따라 네트' metadata={'producer': 'Call PDF v 2.4', 'creator': 'Call PDF', 'creationdate': '', 'title': '', 'author': '', 'subject': '', 'source': '.././data/membersystem_access_rules.pdf', 'total_p

In [123]:
pdf_docs[4].page_content

'- 5 -\n<신설 2025. 12. 15.>⑤ 회원에게 배정하는 통신회선의 용량은 다음 각 호의 범위 내로 한다. <신설 2012. 4. 27., 2015. 4. 13., 2022. 5. 18.,개정 2025. 12. 15.>1. 증권시장의 경우 45M2. 파생상품시장의 경우 45M⑥ 2개 이상의 회원전산센터를 설치한 회원은 각 회원전산센터에 연결된 각 통신회선의 용량이 동일하도록 관리하여야 한다. <신설 2012. 4. 27.,개정 2025. 5. 30.>⑦ 거래소는 회원에게 배정할 수 있는 증권시장과 파생상품시장별 통신회선의 최대 용량을 정하거나 변경하는 경우 이를 즉시 회원에게 통지한다.제6조(통신회선의 신청) ① 회원은 제5조제2항 각 호에서 정한 통신회선의 범위 내에서 거래소에 통신회선의 신규배정, 추가배정 또는 변경배정을 신청할 수 있다. <개정 2012. 4. 27.,2025. 5. 30.>② 제1항에 따라 파생상품시장의 통신회선을 신청하는 회원은 회원전산센터별로 파생상품거래공유회선 또는 파생상품거래공유회선이 아닌 통신회선 중 하나로만 구성되도록 신청하여야 한다. <신설 2025. 5. 30.>③ 제1항의 신청은 별지 제1호 서식에 따라 통신회선 변경일 전 10거래일까지 거래소에 신청한다.④ 회원의 합병·분할·이전 등으로 인하여 회원시스템이 변경되는 경우의 통신회선의 배정 및 신청은 거래소가 그때마다 정하는 바에 따른다. <개정 2015. 8. 7.>제7조(네트워크의 연결) ① 회원은 거래소시스템과 연결하는 회원시스템의 네트워크를 변경하려는 경우 별지 제2호 서식에 따라 네트'

In [124]:
long_text = pdf_docs[4].page_content
print(f'다섯 번째 문서의 텍스트 길이: {len(long_text)}')

다섯 번째 문서의 텍스트 길이: 788


In [125]:
# 각 문서의 텍스트 길이 확인

# 방법 1: 기본 반복문
print("=" * 60)
print("📊 문서별 텍스트 길이")
print("=" * 60)

for i, doc in enumerate(all_docs):
    # i: 인덱스 (0부터 시작)
    # doc: Document 객체
    text_length = len(doc.page_content)
    source = doc.metadata.get("source", "알 수 없음")
    
    print(f"{i+1:2d}. 길이: {text_length:6,d}자 | 출처: {source}")

# 방법 2: 요약 정보
print("\n" + "=" * 60)
print("📈 요약 통계")
print("=" * 60)

# 모든 문서의 길이 리스트
lengths = [len(doc.page_content) for doc in all_docs]

print(f"총 문서 수: {len(all_docs)}개")
print(f"전체 텍스트 길이: {sum(lengths):,}자")
print(f"평균 길이: {sum(lengths) / len(lengths):.0f}자")
print(f"최소 길이: {min(lengths):,}자")
print(f"최대 길이: {max(lengths):,}자")

📊 문서별 텍스트 길이
 1. 길이:    921자 | 출처: .././data/membersystem_access_rules.pdf
 2. 길이:    926자 | 출처: .././data/membersystem_access_rules.pdf
 3. 길이:    819자 | 출처: .././data/membersystem_access_rules.pdf
 4. 길이:    799자 | 출처: .././data/membersystem_access_rules.pdf
 5. 길이:    788자 | 출처: .././data/membersystem_access_rules.pdf
 6. 길이:    870자 | 출처: .././data/membersystem_access_rules.pdf
 7. 길이:    847자 | 출처: .././data/membersystem_access_rules.pdf
 8. 길이:    845자 | 출처: .././data/membersystem_access_rules.pdf
 9. 길이:    765자 | 출처: .././data/membersystem_access_rules.pdf
10. 길이:    564자 | 출처: .././data/membersystem_access_rules.pdf
11. 길이:    621자 | 출처: .././data/membersystem_access_rules.pdf
12. 길이:    463자 | 출처: .././data/membersystem_access_rules.pdf
13. 길이:    190자 | 출처: .././data/membersystem_access_rules.pdf
14. 길이:    137자 | 출처: .././data/membersystem_access_rules.pdf
15. 길이:    338자 | 출처: .././data/membersystem_access_rules.pdf
16. 길이: 251,357자 | 출처: 파생상품시장 업무규정 시행세칙_전문

📈 요약 통계
총 문서 

In [126]:
# TODO: 여기에 청킹 코드를 작성하세요!
# 
# 힌트:
# 1. from langchain_text_splitters import RecursiveCharacterTextSplitter
# 2. text_splitter = RecursiveCharacterTextSplitter(
#        chunk_size=???,
#        chunk_overlap=???,
#        ...
#    )
# 3. chunks = text_splitter.split_documents(all_docs)
# 4. print(f"생성된 청크 수: {len(chunks)}")
#
# 여기서부터 직접 작성해보세요! ↓

- OpenAI text-embedding-3-small 임베딩 모델 사용 (Intel Mac CPU 속도 문제로 bge-m3 대신 선택)
- 같은 모델에 맞는 토크나이저 사용 권장! ⭐⭐⭐⭐⭐ (Claude ai)
  1. 정확성 중요 - 법률 용어 정확히 처리
  2. 긴 문서 - max_length 초과 방지
  3. 프로덕션급 - 나중에 리팩토링 불필요
- text-embedding-3-small → tiktoken (cl100k_base) 사용

In [127]:
# Step 1: 임베딩 모델 결정 (OpenAI - Intel Mac CPU 속도 문제 회피)
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# Step 2: 같은 모델의 토크나이저 가져오기 (tiktoken)
import tiktoken
tokenizer = tiktoken.encoding_for_model("text-embedding-3-small")  # cl100k_base

# 토크나이저 설정 확인
print(f"토크나이저: {tokenizer.name}")  # cl100k_base

# 토큰 수를 계산하는 함수
def count_tokens(text):
    return len(tokenizer.encode(text))

토크나이저: cl100k_base


In [128]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 생성
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                      
    chunk_overlap=100,          
    length_function=count_tokens,         # 토큰 수를 기준으로 분할
    separators=["\n\n", "\n",],           # 구분자 - 재귀적으로 순차적으로 적용 
)

# 텍스트 분할
chunks_raw = text_splitter.split_documents(all_docs)
print(f"분할 전 청크 수: {len(chunks_raw)}")

# 너무 작은 청크 제거 (PDF 페이지 마커 "- 1 -" 등 의미없는 조각 제거)
MIN_TOKENS = 20
chunks = [chunk for chunk in chunks_raw if count_tokens(chunk.page_content) >= MIN_TOKENS]
print(f"필터링 후 청크 수: {len(chunks)}  (제거된 청크: {len(chunks_raw) - len(chunks)}개)")

print(f"\n각 청크의 토큰 수: {[count_tokens(chunk.page_content) for chunk in chunks]}")

분할 전 청크 수: 729
필터링 후 청크 수: 719  (제거된 청크: 10개)

각 청크의 토큰 수: [811, 868, 746, 740, 731, 766, 813, 731, 683, 499, 562, 410, 230, 164, 372, 484, 481, 499, 491, 482, 482, 482, 482, 420, 241, 303, 462, 233, 331, 422, 440, 493, 306, 392, 364, 426, 476, 189, 255, 446, 265, 336, 118, 489, 199, 458, 474, 140, 465, 471, 389, 472, 366, 481, 183, 480, 145, 397, 224, 379, 493, 487, 200, 457, 271, 403, 309, 379, 488, 485, 466, 479, 250, 311, 323, 477, 343, 277, 310, 346, 429, 225, 400, 139, 452, 306, 349, 450, 73, 481, 151, 186, 315, 430, 432, 490, 77, 472, 451, 285, 485, 108, 441, 338, 365, 418, 411, 416, 89, 282, 448, 447, 452, 229, 411, 496, 264, 468, 321, 326, 101, 428, 477, 423, 385, 277, 430, 271, 206, 432, 194, 322, 247, 453, 362, 417, 329, 490, 487, 488, 343, 470, 228, 443, 443, 428, 465, 478, 224, 358, 485, 448, 239, 414, 91, 213, 329, 443, 165, 489, 306, 434, 374, 359, 467, 268, 439, 120, 454, 388, 439, 382, 421, 402, 208, 261, 390, 370, 489, 123, 377, 214, 447, 475, 397, 121, 454, 478, 143,

In [129]:
# 청크의 텍스트 확인
print(chunks[20].page_content)

개정 2016. 6. 20. 규정 제1337호
개정 2016. 7. 12. 규정 제1349호
개정 2016. 8. 31. 규정 제1355호
개정 2016. 9. 6. 규정 제1358호
개정 2016. 11. 21. 규정 제1379호
개정 2017. 3. 13. 규정 제1430호
개정 2017. 5. 19. 규정 제1452호
개정 2017. 6. 15. 규정 제1460호
개정 2017. 9. 19. 규정 제1492호
개정 2017. 9. 20. 규정 제1493호
개정 2017. 11. 10. 규정 제1497호
개정 2018. 2. 28. 규정 제1534호
개정 2018. 3. 15. 규정 제1541호
개정 2018. 3. 23. 규정 제1547호
개정 2018. 6. 12. 규정 제1565호
개정 2018. 8. 17. 규정 제1576호
개정 2018. 9. 7. 규정 제1578호
개정 2018. 12. 5. 규정 제1600호
개정 2018. 12. 7. 규정 제1610호
개정 2019. 1. 25. 규정 제1635호
개정 2019. 4. 4. 규정 제1659호
개정 2019. 7. 1. 규정 제1692호
개정 2019. 7. 22. 규정 제1707호


In [130]:
# 청크의 텍스트 확인
print(chunks[20].metadata)

{'source': '파생상품시장 업무규정 시행세칙_전문'}


## 3. 벡터 저장소 기반 RAG 검색기 (Retriever)


### `3-(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [131]:
# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}") # 1536 (text-embedding-3-small)

임베딩 차원: 1536


In [132]:
# OpenAI 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = 1536  # text-embedding-3-small 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)  
print("=== FAISS 인덱스 초기화 완료")

# FAISS 벡터 저장소의 벡터 차원 수 (임베딩 차원 수)
faiss_index.d

=== FAISS 인덱스 초기화 완료


1536

In [133]:
# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 메모리에 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

0

In [134]:
import uuid

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(chunks))]

# 문서를 벡터 저장소에 저장
# 힌트: faiss_db.add_documents(chunks, ids=doc_ids) 사용
added_doc_ids = faiss_db.add_documents(chunks, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"=== {len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(added_doc_ids)

=== 719개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['f4d731a7-90ce-44c7-9fda-fbaf2849fbf8', 'd82fb888-c6d4-43e9-b357-264fb1c591a8', 'd18627e6-d07d-44c3-a4ac-4a484c5a1505', '1b1eb3ef-9a00-41f1-abaf-5bb02b4e97be', '203baf39-65f5-4045-8914-77e1113f5412', 'e0134443-5a4b-4533-974f-5012af8d445c', '20bfdde7-dc2e-4d5a-8fa0-51a85d616837', '7d0b5395-9d85-4678-8080-c98828261df5', '4070fdcf-4e0e-49aa-8409-aa731c7fad23', '75dc3c6c-f005-49ca-ac32-1aa466b4d834', '649a1afd-775e-432d-9530-141cc22434e1', 'a7053bc2-f814-44d2-a4d5-07310e9f07be', '62ef5dda-961d-48ca-9104-76558fcdf8bc', 'f8fbd8f7-8cd2-4aee-9637-d85b4abe5aaf', '37d1bd2a-4be4-4307-b50e-f3ab87f488d1', '15aaca23-828a-4de6-8f14-0fe25070d7fc', '84541278-c39a-4abf-a6e4-5afb388d58db', 'c556325b-7149-4956-a047-1e9543ce3cec', '269dce55-5c04-4184-b848-e0c17d2b377f', 'f6700a1f-fd50-4c37-9291-2a9401e6f343', 'b4a26140-6e1b-4c86-bbbb-7c99e90a5cd4', '69e007cb-e4bb-4bc4-ab95-afea614c2cd8', 'beb3ba0b-c9b9-4bb9-a611-60b3f1aa59e3', 'd9fa9b12-4c29-4a6e-8361-0c3ffacbbe5f', '9

In [135]:
# 저장된 문서의 갯수 확인
faiss_db.index.ntotal

719

In [136]:
# 저장된 인덱스 확인
faiss_db.index_to_docstore_id

{0: 'f4d731a7-90ce-44c7-9fda-fbaf2849fbf8',
 1: 'd82fb888-c6d4-43e9-b357-264fb1c591a8',
 2: 'd18627e6-d07d-44c3-a4ac-4a484c5a1505',
 3: '1b1eb3ef-9a00-41f1-abaf-5bb02b4e97be',
 4: '203baf39-65f5-4045-8914-77e1113f5412',
 5: 'e0134443-5a4b-4533-974f-5012af8d445c',
 6: '20bfdde7-dc2e-4d5a-8fa0-51a85d616837',
 7: '7d0b5395-9d85-4678-8080-c98828261df5',
 8: '4070fdcf-4e0e-49aa-8409-aa731c7fad23',
 9: '75dc3c6c-f005-49ca-ac32-1aa466b4d834',
 10: '649a1afd-775e-432d-9530-141cc22434e1',
 11: 'a7053bc2-f814-44d2-a4d5-07310e9f07be',
 12: '62ef5dda-961d-48ca-9104-76558fcdf8bc',
 13: 'f8fbd8f7-8cd2-4aee-9637-d85b4abe5aaf',
 14: '37d1bd2a-4be4-4307-b50e-f3ab87f488d1',
 15: '15aaca23-828a-4de6-8f14-0fe25070d7fc',
 16: '84541278-c39a-4abf-a6e4-5afb388d58db',
 17: 'c556325b-7149-4956-a047-1e9543ce3cec',
 18: '269dce55-5c04-4184-b848-e0c17d2b377f',
 19: 'f6700a1f-fd50-4c37-9291-2a9401e6f343',
 20: 'b4a26140-6e1b-4c86-bbbb-7c99e90a5cd4',
 21: '69e007cb-e4bb-4bc4-ab95-afea614c2cd8',
 22: 'beb3ba0b-c9b9-

In [137]:
# 저장된 문서 검색
faiss_db.docstore.search('DOC_1')

'ID DOC_1 not found.'

### `3-(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [ ]:
# mmr 검색기 생성
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k': 3,
        'fetch_k': 20,       # 후보 더 많이 가져오기 (10 → 20)
        'lambda_mult': 0.7   # 관련성 비중 높이기 (0.3 → 0.7)
    }
)

In [139]:
# 검색 테스트 
query = "통신회선을 신청할 때 몇 거래일 전까지 신청해야 하나요?"
# 힌트: faiss_mmr_retriever.invoke(query) 사용
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"-{i}-\n{doc.page_content[:100]}...{doc.page_content[-100:]}")
    print("-" * 100)

쿼리: 통신회선을 신청할 때 몇 거래일 전까지 신청해야 하나요?
검색 결과:
-1-
- 13 -
[별지제1호서식]통신회선신규·추가·변경신청서<개정2025.5.30.>통신회선신규·추가·변경신청서구분신청내역비고회원명신청내용대역폭설치장소테스트일(예정)가동일(예정)연락처...가동일(예정)연락처∙담당자명:(인)∙소속부서:∙전화번호:∙E-mail:비고.파생상품시장의경우정규거래·야간거래회선을구분(파생상품거래공유회선여부등포함)하여신청신청일:년월일한국거래소귀하
----------------------------------------------------------------------------------------------------
-2-
제11조(시장조성자의 주권연계계좌 신고에 대한 특례) 제83조제3항의 개정규정에도 불구하고 부칙 제1조 본문의 시행일 이전의 섹터지수선물·주식선물·주식옵션거래 시장조성자는 부칙 제...도 불구하고 부칙 제1조 본문의 시행일 이전의 섹터지수선물·주식선물·주식옵션거래 시장조성자는 부칙 제1조 본문의 시행일부터 5거래일 이내에 주권연계계좌를 거래소에 신고하여야 한다.
----------------------------------------------------------------------------------------------------
-3-
⑦ 거래소는 대량투자자착오거래의 구제와 관련된 서류를 10년간 기록·유지한다.<개정 2014. 8. 20.>
[전문개정 ]...⑦ 거래소는 대량투자자착오거래의 구제와 관련된 서류를 10년간 기록·유지한다.<개정 2014. 8. 20.>
[전문개정 ]
----------------------------------------------------------------------------------------------------


### `3-(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [140]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain_core.prompts import ChatPromptTemplate

template = """당신은 KRX(한국거래소) 현행 규정 전문가입니다.

[작업 지침]
- 반드시 아래 [컨텍스트]만을 근거로 답변하세요.
- 외부 지식 사용은 제한하세요.
- 아래 [컨텍스트]에 근거했을 때 불확실한 경우 불확실하다고 답변하세요.
- 답변 불가능한 경우 답변 불가능하다고 답변하세요. 이 때 답변 불가능한 구체적이고 논리적인 이유를 함께 제시하세요.
- 추측이나 가정은 최소화하세요.
- 명확한 근거 제시가 필요합니다.
- 구조화된 형태로 작성되어야 합니다.
- 한국어로 답변하세요.

[컨텍스트]
{context}

[질문] 
{question}

[답변 형식 가이드]
- 핵심 답변 섹션: 주요 내용을 요약하여 핵심만 정리합니다.
- 근거 제시 섹션: 위 핵심 답변 섹션에 언급한 각 항목에 대해 [컨텍스트]에서 찾은 모든 내용을 정확하고 구체적으로 인용하세요. 구조화된 bullet point 형식으로 답변하세요.
- 추가 안내 섹션(필요시): 질문에 완전히 답변하려면 어떤 추가 문서나 규정을 확인해야 하는지 안내해 주세요. 외부 지식을 직접 제공하지 말고, 참조해야 할 문서/규정명만 안내하세요.
"""

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

================================ Human Message =================================

당신은 KRX(한국거래소) 현행 규정 전문가입니다.

[작업 지침]
- 반드시 아래 [컨텍스트]만을 근거로 답변하세요.
- 외부 지식 사용은 제한하세요.
- 아래 [컨텍스트]에 근거했을 때 불확실한 경우 불확실하다고 답변하세요.
- 답변 불가능한 경우 답변 불가능하다고 답변하세요. 이 때 답변 불가능한 구체적이고 논리적인 이유를 함께 제시하세요.
- 추측이나 가정은 최소화하세요.
- 명확한 근거 제시가 필요합니다.
- 구조화된 형태로 작성되어야 합니다.
- 한국어로 답변하세요.

[컨텍스트]
{context}

[질문] 
{question}

[답변 형식 가이드]
- 핵심 답변 섹션: 주요 내용을 요약하여 핵심만 정리합니다.
- 근거 제시 섹션: 위 핵심 답변 섹션에 언급한 각 항목에 대해 [컨텍스트]에서 찾은 모든 내용을 정확하고 구체적으로 인용하세요. 구조화된 bullet point 형식으로 답변하세요.
- 추가 안내 섹션(필요시): 질문에 완전히 답변하려면 어떤 추가 문서나 규정을 확인해야 하는지 안내해 주세요. 외부 지식을 직접 제공하지 말고, 참조해야 할 문서/규정명만 안내하세요.



### `3-(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그대로 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4o-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성

In [141]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
# 힌트: ChatOpenAI(model='gpt-4o-mini', temperature=0) 사용
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 문서 포맷팅 (출처 정보 포함 → LLM이 근거 제시 섹션에서 출처 활용 가능)
def format_docs(docs):
    return "\n\n".join([
        f"[출처: {doc.metadata.get('source', '불명')}]\n{doc.page_content}"
        for doc in docs
    ])

# RAG 체인 생성
# 힌트: {'context': faiss_mmr_retriever | format_docs, 'question': RunnablePassthrough()} | prompt | llm | StrOutputParser()
rag_chain = (
    {
        'context': faiss_mmr_retriever | format_docs, 
        'question': RunnablePassthrough()
    }
    | prompt 
    | llm 
    | StrOutputParser()
)

# 체인 실행
query = "통신회선을 신청할 때 몇 거래일 전까지 신청해야 하나요?"
output = rag_chain.invoke(query)

print(f"쿼리: {query}")
print("답변:")
print(output)

쿼리: 통신회선을 신청할 때 몇 거래일 전까지 신청해야 하나요?
답변:
### 핵심 답변
통신회선을 신청할 때 특정 거래일 전까지 신청해야 한다는 정보는 [컨텍스트]에 명시되어 있지 않습니다. 따라서 정확한 신청 기한에 대한 정보는 불확실합니다.

### 근거 제시
- [컨텍스트]에는 통신회선 신청에 대한 구체적인 거래일 전 신청 기한에 대한 내용이 포함되어 있지 않음.
- 통신회선 신청서의 내용은 대역폭, 설치 장소, 테스트일, 가동일 등과 관련된 정보만 포함되어 있음.

### 추가 안내
정확한 통신회선 신청 기한을 확인하기 위해서는 한국거래소의 관련 규정이나 지침을 참조해야 합니다.


### `3-(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [144]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    
    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response

# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=get_streaming_response,
    title="KRX 규정 질의응답 시스템",
    description="한국거래소(KRX) 현행 규정을 기반으로 질의응답합니다.\n\n📄 참조 문서: 회원시스템접속등에관한지침 | 파생상품시장 업무규정 시행세칙",
    examples=[
        "통신회선을 신청할 때 몇 거래일 전까지 신청해야 하나요?",
        "거래소시스템이란 무엇인가요?",
        "기초자산기준가격이란 무엇인가요?",
        "의제약정가격의 정의를 설명해주세요.",
        "파생상품시장에서 회원이 지켜야 할 통신 관련 규정은 무엇인가요?",
    ],
    type="messages",
)

# 실행 (포트 명시 + 인라인 미리보기)
demo.launch()

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [145]:
# demo 실행 종료
demo.close()

Closing server running on port: 7860
